In [ ]:
import logging
import boto3
from botocore.exceptions import ClientError

# Configuración del estándar de logs
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Parámetros de configuración
BUCKET_NAME = ''
PREFIX = ''

def list_s3_weather_files(bucket_name: str, prefix: str) -> None:
    """
    Lista los archivos climáticos en formato CSV almacenados en un bucket de Amazon S3.
    La autenticación se realiza de manera implícita mediante el IAM Instance Profile.
    """
    s3_client = boto3.client('s3', region_name='us-east-1')
    
    try:
        logger.info(f"Iniciando solicitud de listado en S3. Bucket: {bucket_name} | Prefijo: {prefix}")
        response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
        
        if 'Contents' not in response:
            logger.warning(f"La ruta s3://{bucket_name}/{prefix} no contiene objetos.")
            return

        # Filtrado de archivos con extensión CSV
        csv_files = [obj for obj in response['Contents'] if obj['Key'].endswith('.csv')]
        
        if not csv_files:
            logger.warning(f"No se encontraron archivos con extensión .csv bajo el prefijo especificado.")
            return
            
        logger.info(f"Listado completado con éxito. Se encontraron {len(csv_files)} archivos CSV:")
        for file in csv_files:
            print(f"  - Key: {file['Key']} | Size: {file['Size']} bytes")
            
    except ClientError as aws_error:
        # Captura errores específicos de AWS (permisos insuficientes, bucket inexistente, etc.)
        error_code = aws_error.response['Error']['Code']
        error_message = aws_error.response['Error']['Message']
        logger.error(f"Error de AWS SDK [{error_code}]: {error_message}")
        
    except Exception as e:
        # Captura errores generales del sistema o de red
        logger.error(f"Error inesperado durante la ejecución del proceso: {str(e)}")

if __name__ == "__main__":
    list_s3_weather_files(BUCKET_NAME, PREFIX)

2026-08-26 03:09:17,704 - INFO - Found credentials from IAM Role: LabRole
2026-08-26 03:09:18,075 - INFO - Iniciando solicitud de listado en S3. Bucket: s3-bucket-cloud-computing-unisabana-fjaa | Prefijo: raw-zone/weather/
2026-08-26 03:09:18,250 - INFO - Listado completado con éxito. Se encontraron 12 archivos CSV:


  - Key: raw-zone/weather/weather-data_20260822164729.csv | Size: 115 bytes
  - Key: raw-zone/weather/weather-data_20260822170003.csv | Size: 115 bytes
  - Key: raw-zone/weather/weather-data_20260822173402.csv | Size: 115 bytes
  - Key: raw-zone/weather/weather-data_20260822180003.csv | Size: 114 bytes
  - Key: raw-zone/weather/weather-data_20260822183402.csv | Size: 114 bytes
  - Key: raw-zone/weather/weather-data_20260826000007.csv | Size: 118 bytes
  - Key: raw-zone/weather/weather-data_20260826003403.csv | Size: 119 bytes
  - Key: raw-zone/weather/weather-data_20260826010004.csv | Size: 120 bytes
  - Key: raw-zone/weather/weather-data_20260826013403.csv | Size: 118 bytes
  - Key: raw-zone/weather/weather-data_20260826020005.csv | Size: 119 bytes
  - Key: raw-zone/weather/weather-data_20260826023403.csv | Size: 117 bytes
  - Key: raw-zone/weather/weather-data_20260826030004.csv | Size: 117 bytes


In [ ]:
import io
import logging
import boto3
import pandas as pd
from botocore.exceptions import ClientError

# Configuración estandarizada para salida de logs técnicos
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Parámetros de ruta en el Data Lake
BUCKET_NAME = ''
PREFIX = ''

def consolidate_weather_data(bucket_name: str, prefix: str) -> pd.DataFrame:
    """
    Descarga en memoria los archivos CSV de clima desde Amazon S3,
    los unifica en un único DataFrame de Pandas y calcula estadísticas descriptivas.
    """
    s3_client = boto3.client('s3', region_name='us-east-1')
    dataframes = []
    
    try:
        logger.info(f"Buscando objetos climáticos bajo el prefijo: s3://{bucket_name}/{prefix}")
        response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
        
        if 'Contents' not in response:
            logger.warning("La consulta S3 no retornó objetos para el prefijo indicado.")
            return pd.DataFrame()
            
        csv_files = [obj for obj in response['Contents'] if obj['Key'].endswith('.csv')]
        
        if not csv_files:
            logger.warning("No se identificaron archivos con extensión .csv en el bucket.")
            return pd.DataFrame()
            
        logger.info(f"Se iniciará el procesamiento en memoria de {len(csv_files)} archivos climáticos.")
        
        # Iteración de objetos y lectura directa en memoria (bypassing almacenamiento local)
        for file in csv_files:
            key = file['Key']
            try:
                # Recuperación del objeto de S3
                obj_response = s3_client.get_object(Bucket=bucket_name, Key=key)
                file_content = obj_response['Body'].read()
                
                # Carga secuencial a DataFrame de pandas vía io.BytesIO
                df_temp = pd.read_csv(io.BytesIO(file_content))
                dataframes.append(df_temp)
            except ClientError as e:
                logger.error(f"Error de SDK al recuperar el objeto {key}: {e.response['Error']['Message']}")
            except Exception as e:
                logger.error(f"Error general procesando el archivo {key}: {str(e)}")
        
        if not dataframes:
            logger.warning("Ningún archivo pudo ser cargado a memoria.")
            return pd.DataFrame()
            
        # Unificación de estructuras de datos tabulares parciales
        consolidated_df = pd.concat(dataframes, ignore_index=True)
        logger.info(f"Consolidación exitosa. DataFrame estructurado con {len(consolidated_df)} filas.")
        
        return consolidated_df
        
    except ClientError as aws_error:
        logger.error(f"Error crítico en AWS SDK [{aws_error.response['Error']['Code']}]: {aws_error.response['Error']['Message']}")
        return pd.DataFrame()
    except Exception as e:
        logger.error(f"Excepción general de ejecución: {str(e)}")
        return pd.DataFrame()

def generate_weather_summary(df: pd.DataFrame) -> None:
    """
    Calcula y muestra el resumen de métricas climáticas requeridas para la analítica.
    """
    if df.empty:
        logger.warning("El DataFrame consolidado está vacío. Omitiendo generación de estadísticas.")
        return
        
    # Validar que existan las columnas de interés descritas en el entregable
    required_cols = ['temperature_C', 'humidity_pct']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        logger.error(f"Estructura incompatible. Columnas faltantes: {missing_cols}")
        return
        
    # Obtención de métricas agregadas
    total_readings = len(df)
    avg_temp = df['temperature_C'].mean()
    min_humidity = df['humidity_pct'].min()
    max_humidity = df['humidity_pct'].max()
    
    print("\n" + "="*60)
    print("      RESUMEN ESTADÍSTICO DE DATOS CLIMÁTICOS CONSOLIDADOS")
    print("="*60)
    print(f" Total de registros procesados : {total_readings}")
    print(f" Temperatura promedio (ºC)     : {avg_temp:.2f} °C")
    print(f" Humedad relativa mínima (%)   : {min_humidity:.1f} %")
    print(f" Humedad relativa máxima (%)   : {max_humidity:.1f} %")
    print("="*60)

if __name__ == "__main__":
    df_clima = consolidate_weather_data(BUCKET_NAME, PREFIX)
    if not df_clima.empty:
        print("\nVista previa de los datos unificados:")
        print(df_clima.head())
        generate_weather_summary(df_clima)

2026-08-26 03:24:27,941 - INFO - Buscando objetos climáticos bajo el prefijo: s3://s3-bucket-cloud-computing-unisabana-fjaa/raw-zone/weather/
2026-08-26 03:24:28,011 - INFO - Se iniciará el procesamiento en memoria de 12 archivos climáticos.
2026-08-26 03:24:28,748 - INFO - Consolidación exitosa. DataFrame estructurado con 12 filas.



Vista previa de los datos unificados:
       extraction_date  temperature_C  humidity_pct weather_desc  \
0  2026-08-22 16:47:29          14.89            88   light rain   
1  2026-08-22 17:00:03          14.89            88   light rain   
2  2026-08-22 17:34:02          15.22            88   light rain   
3  2026-08-22 18:00:03          14.90            87   light rain   
4  2026-08-22 18:34:02          14.90            88   light rain   

   wind_speed_ms  
0           1.97  
1           1.97  
2           1.97  
3           1.12  
4           1.36  

      RESUMEN ESTADÍSTICO DE DATOS CLIMÁTICOS CONSOLIDADOS
 Total de registros procesados : 12
 Temperatura promedio (ºC)     : 14.90 °C
 Humedad relativa mínima (%)   : 55.0 %
 Humedad relativa máxima (%)   : 88.0 %
